# Brain tumor classifier: honest held-out evaluation

Trains the baseline CNN, ResNet50 and EfficientNetB0 on the dataset's official **Training** split
only, then evaluates each one on the official **Testing** split. Any test image that is
byte-identical to a training image is dropped first, so no test scan is ever seen in training.

**Setup (one time):** in the right-hand panel, set *Accelerator* to **GPU T4 x2** (or P100),
turn **Internet on**, and use *Add Input* to add the dataset
**masoudnickparvar/brain-tumor-mri-dataset**. Then choose *Run All*. It takes about 15–25 minutes.

When it finishes, download `/kaggle/working/results.zip` from the *Output* panel.

In [ ]:
!git clone -q -b honest-eval https://github.com/satyajit7018/brain-tumor-classifier.git
%cd brain-tumor-classifier
!pip install -q reportlab
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Copy the Kaggle dataset into data/raw/<class>/, prefixing each file with its
# original split (Training_ / Testing_), which is what split_by_source() reads.
import glob, os
from pathlib import Path
from scripts.download_dataset import organize_extracted_data
from src.data.dataset import split_by_source, CLASS_NAMES

src = Path(next(p for p in glob.glob("/kaggle/input/**/Training", recursive=True))).parent
print("Dataset found at", src)
organize_extracted_data(src, Path("data/raw"))

train_items, test_items, dropped = split_by_source("data/raw")
print(f"train pool: {len(train_items)}  held-out test: {len(test_items)}  "
      f"test images dropped as duplicates of training images: {dropped}")

In [ ]:
!rm -rf docs/gradcam_examples && mkdir -p docs/gradcam_examples
for m in ["baseline_cnn", "efficientnet_b0", "resnet50"]:
    print(f"\n========== {m} ==========")
    !python scripts/train_final.py --model {m} --epochs 25 --output-model saved_models/{m}.keras
    gc_dir = "docs/gradcam_examples" if m == "resnet50" else f"/tmp/gradcam_{m}"
    !python scripts/evaluate_final.py --model-path saved_models/{m}.keras --output-json docs/eval_results_{m}.json --gradcam-dir {gc_dir}

In [ ]:
import json
print(f"{'model':<17}{'n_test':>7}{'accuracy':>10}{'macro F1':>10}{'FNR':>8}{'mean AUC':>9}")
for m in ["baseline_cnn", "efficientnet_b0", "resnet50"]:
    r = json.load(open(f"docs/eval_results_{m}.json"))
    rep, aucs = r["classification_report"], [v for v in r["roc_auc_per_class"].values() if v is not None]
    print(f"{m:<17}{r['evaluation_split']['n_test']:>7}{rep['accuracy']:>10.4f}"
          f"{rep['macro avg']['f1-score']:>10.4f}{r['false_negative_rate']:>8.4f}{sum(aucs)/len(aucs):>9.4f}")

In [ ]:
!cp saved_models/*_history.json docs/ 2>/dev/null; cp saved_models/resnet50.keras /kaggle/working/
!cd docs && zip -qr /kaggle/working/results.zip eval_results_*.json *_history.json gradcam_examples
!ls -la /kaggle/working